In [1]:
#all imports needed for EDA
from huggingface_hub import hf_hub_download
import duckdb
import pandas as pd
import numpy as np
import tarfile

In [2]:
repo_name = "AmazonScience/WikiDT"

split_files = {
    "train" : "WikiDT-dataset/samples/train.json",
    "val" : "WikiDT-dataset/samples/val.json",
    "test" : "WikiDT-dataset/samples/test.json"
}


In [3]:
combined_data = {}
for split_name, file_path in split_files.items():
    try:
        path = hf_hub_download(repo_id=repo_name, filename=file_path, repo_type="dataset")
        combined_data[split_name] = path
    except Exception:
        print(f"Failed to download {file_path} from {repo_name}. Trying next split...")
        continue

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [4]:
ocr_tar_path = hf_hub_download(
    repo_id=repo_name,
    filename="WikiDT-dataset/ocr.tar",
    repo_type="dataset",
)

In [5]:
con = duckdb.connect()

stage_info = []

for split_name, file_path in combined_data.items():
    print(f"Loading {split_name} from {file_path}...")
    stage_split_info = con.execute(f"""
        SELECT
            sample_id,
            image,
            question,
            json_extract_string(answer, '$[0][0]') AS answer,
            ocr_retrieval_file_textract,
            all_ocr_files_textract,
            '{split_name}' AS original_split
        FROM read_json(
            '{file_path}',
            columns={{
                'sample_id': 'VARCHAR',
                'image': 'VARCHAR',
                'question': 'VARCHAR',
                'answer': 'JSON',
                'ocr_retrieval_file_textract': 'VARCHAR',
                'all_ocr_files_textract': 'VARCHAR[]'
            }}
        )
    """).df()
    stage_info.append(stage_split_info)

df_all = pd.concat(stage_info, ignore_index=True)
total_combined_rows = len(df_all)
print(f"Total pooled QA pairs loaded: {total_combined_rows:,}")


Loading train from /root/.cache/huggingface/hub/datasets--AmazonScience--WikiDT/snapshots/db803a4c2031b7170063fed7bc0c445cb8aec9f6/WikiDT-dataset/samples/train.json...
Loading val from /root/.cache/huggingface/hub/datasets--AmazonScience--WikiDT/snapshots/db803a4c2031b7170063fed7bc0c445cb8aec9f6/WikiDT-dataset/samples/val.json...
Loading test from /root/.cache/huggingface/hub/datasets--AmazonScience--WikiDT/snapshots/db803a4c2031b7170063fed7bc0c445cb8aec9f6/WikiDT-dataset/samples/test.json...
Total pooled QA pairs loaded: 70,652


In [6]:
print("Indexing members in ocr.tar...")
with tarfile.open(ocr_tar_path, "r") as tar:
    tar_members = set(tar.getnames())

print('tar_members', tar_members)

available_textract_files = set()
for m in tar_members:
    clean = m.lstrip("./")
    if clean.startswith("WikiDT-dataset/"):
        clean = clean.replace("WikiDT-dataset/", "")
    if clean.startswith("ocr/textract/"):
        available_textract_files.add(clean)

print(f"Total available Textract OCR files in archive: {len(available_textract_files):,}")

Indexing members in ocr.tar...
tar_members {'ocr/web/11999122_page_seg_3.json', 'ocr/textract/10634933_page_seg_9.json', 'ocr/textract/15537596_page_seg_13.json', 'ocr/textract/22603701_page_seg_0.json', 'ocr/textract/20468206_page_seg_1.json', 'ocr/textract/14747235_page_seg_1.json', 'ocr/web/15597975_page_seg_3.json', 'ocr/web/1516353_page_seg_2.json', 'ocr/textract/10040473_page_seg_1.json', 'ocr/web/272313_page_seg_0.json', 'ocr/web/17239112_page_seg_1.json', 'ocr/textract/10429820_page_seg_2.json', 'ocr/textract/14027795_page_seg_2.json', 'ocr/web/12427499_page_seg_2.json', 'ocr/web/1415821_page_seg_4.json', 'ocr/textract/204_925_page_seg_0.json', 'ocr/textract/11823140_page_seg_0.json', 'ocr/web/17162228_page_seg_2.json', 'ocr/textract/204_784_page_seg_0.json', 'ocr/textract/18633729_page_seg_1.json', 'ocr/textract/15026994_page_seg_3.json', 'ocr/textract/20990052_page_seg_1.json', 'ocr/web/1816948_page_seg_1.json', 'ocr/web/18252006_page_seg_6.json', 'ocr/textract/1140114_page_s

In [7]:
def audit_row_coverage(row):
    primary = str(row["ocr_retrieval_file_textract"]).lstrip("./")
    if primary in available_textract_files:
        return "primary_match", primary

    fallbacks = row["all_ocr_files_textract"]
    if isinstance(fallbacks, (list, np.ndarray)):
        for fb in fallbacks:
            clean_fb = str(fb).lstrip("./")
            if clean_fb in available_textract_files:
                return "fallback_match", clean_fb

    return "missing", None

audit_results = df_all.apply(audit_row_coverage, axis=1)
df_all["mapping_status"] = [res[0] for res in audit_results]
df_all["resolved_ocr_path"] = [res[1] for res in audit_results]

In [8]:
statuses = [res[0] for res in audit_results]
primary_count = statuses.count("primary_match")
fallback_count = statuses.count("fallback_match")
missing_count = statuses.count("missing")
total_rows = len(statuses)

print("=" * 55)
print("=== OCR Mapping Resolution Breakdown ===")
print("=" * 55)
print(f"Primary Target Matches:   {primary_count:>7,} ({primary_count / total_rows * 100:>6.2f}%)")
print(f"Fallback Segment Matches: {fallback_count:>7,} ({fallback_count / total_rows * 100:>6.2f}%)")
print(f"Missing (Dropped):        {missing_count:>7,} ({missing_count / total_rows * 100:>6.2f}%)")
print("-" * 55)
print(f"Total Usable OCR Rows:    {primary_count + fallback_count:>7,} ({(primary_count + fallback_count) / total_rows * 100:>6.2f}%)")
print("=" * 55)

=== OCR Mapping Resolution Breakdown ===
Primary Target Matches:    69,316 ( 98.11%)
Fallback Segment Matches:   1,209 (  1.71%)
Missing (Dropped):            127 (  0.18%)
-------------------------------------------------------
Total Usable OCR Rows:     70,525 ( 99.82%)


In [9]:
df_filtered = df_all[df_all["mapping_status"] != "missing"].copy().reset_index(drop=True)
total_retained_rows = len(df_filtered)

print("\n" + "=" * 65)
print("=== Complete Dataset OCR Coverage Summary ===")
print("=" * 65)
print(f"Total Initial Pooled Rows:     {total_combined_rows:>8,}")
print(f"Retained (Valid OCR Target):   {total_retained_rows:>8,} ({total_retained_rows / total_combined_rows * 100:>6.2f}%)")
print(f"Dropped (Missing OCR):         {total_combined_rows - total_retained_rows:>8,} ({(total_combined_rows - total_retained_rows) / total_combined_rows * 100:>6.2f}%)")
print(f"Unique Retained Documents:     {df_filtered['image'].nunique():>8,}")
print("=" * 65)


=== Complete Dataset OCR Coverage Summary ===
Total Initial Pooled Rows:       70,652
Retained (Valid OCR Target):     70,525 ( 99.82%)
Dropped (Missing OCR):              127 (  0.18%)
Unique Retained Documents:       14,148
